In [1]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import gc
import torch
from torch import nn
import os
import getpass
import copy
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassRecall
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device = {str(device).title()}")

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
df_path = os.path.join(script_dir,"Network Datasets","Customer Segmentation Dataset.csv")
df = pd.read_csv(df_path)

df = df.dropna()

x = df.drop(columns="loyalty_segment").values
encoder = LabelEncoder()
y = encoder.fit_transform(df["loyalty_segment"])

x_scaler = StandardScaler()
x = x_scaler.fit_transform(x)

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=12)

print(f"X = {x.shape}")
print(f"Y = {y.shape}")
print(f"X Train = {x_train.shape}")
print(f"Y Train = {y_train.shape}")

num_features = x_train.shape[1]
num_classes = len(encoder.classes_)

x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)
x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.long).to(device)

user_account = getpass.getuser()

class MultiClassClassifier(nn.Module):
    def __init__(self, input_dim=num_features, out_dim=num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=input_dim, out_features=8),
            nn.LeakyReLU(),
            nn.BatchNorm1d(8),

            nn.Linear(in_features=8, out_features=16),
            nn.LeakyReLU(),
            nn.BatchNorm1d(16),
            
            nn.Linear(in_features=16, out_features=28),
            nn.LeakyReLU(),
            nn.BatchNorm1d(28),

            nn.Linear(in_features=28, out_features=47),
            nn.LeakyReLU(),
            nn.BatchNorm1d(47),
            nn.Dropout(p=0.7),   
            
            nn.Linear(in_features=47, out_features=52),
            nn.ELU(),
            nn.BatchNorm1d(52),
            nn.Dropout(p=0.8),   

            nn.Linear(in_features=52, out_features=68),
            nn.ELU(),
            nn.BatchNorm1d(68),
            nn.Dropout(p=0.3),

            nn.Linear(in_features=68, out_features=42),
            nn.LeakyReLU(),
            nn.BatchNorm1d(42),   

            nn.Linear(in_features=42, out_features=21),
            nn.LeakyReLU(),
            nn.BatchNorm1d(21),

            nn.Linear(in_features=21, out_features=out_dim)
        )     

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        return self.network(x)

def run_grid_search_multiclass(rates_list, epochs_list, seed_val):
    best_loss = float("inf")
    best_state = None
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    accuracies = []

    print(f"\n== Search Leaderboard ==")
    print(f"{'Learning Rate':<15} | {'Epochs':<8} | {'Val Loss':<10} | {'Val Acc %':<10}")
    print("=" * 65)

    for lr in rates_list:
        for epoch in epochs_list:
            torch.manual_seed(int(seed_val))
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(int(seed_val))
            
            model = MultiClassClassifier().to(device)   
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)

            for e in tqdm(range(epoch)):
                model.train()
                y_preds = model(x_train)
                loss = loss_fn(y_preds, y_train.squeeze().long())     

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.inference_mode():
                test_logits = model(x_test)
                test_loss = loss_fn(test_logits, y_test.squeeze().long()).item()
                metric_calc = MulticlassAccuracy(num_classes=num_classes).to(device)
                test_acc = metric_calc(test_logits, y_test.squeeze().long()).item()
                losses.append(test_loss)
                accuracies.append(test_acc)

            print(f"{lr:<15} | {epoch:<8} | {test_loss:<10.4f} | {test_acc * 100} %")

            if test_loss < best_loss:
                best_loss = test_loss
                best_state = copy.deepcopy(model.state_dict()) 

            del model, optimizer
            gc.collect()       
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\n● Best Loss Achieved = {best_loss:<10.4f}")  
    print(f"● Worst Loss = {max(losses):<10.4f}")     
    print(f"● Mean Accuracy = {sum(accuracies) / len(accuracies) * 100} %")

    precision_calc = MulticlassPrecision(num_classes=num_classes).to(device)
    recall_calc = MulticlassRecall(num_classes=num_classes).to(device)

    print(f"● Precision Score = {precision_calc(test_logits, y_test.squeeze().long()).item():.4f}")
    print(f"● Recall Score = {recall_calc(test_logits, y_test.squeeze().long()).item():.4f}\n")


    return best_state, best_loss, losses

epochs_list = [450, 610, 790, 820]
rates = [0.01, 0.001, 0.1, 0.2]

for file in [f"C:/Users/{user_account}/OneDrive/Desktop/Customer Segmentation Status Multi Classifer Network Details.txt"]:
    if os.path.exists(file):
        try:os.remove(file)
        except OSError:pass

seed = float(input("-- Enter a Value for a Seed : ").strip())
best_state, best_loss, losses = run_grid_search_multiclass(rates, epochs_list, seed)
torch.save(best_state, f"C:/Users/{user_account}/OneDrive/Desktop/Customer Segmentation Status Multi Classifer Network Parameters")
fresh_model = MultiClassClassifier().to(device)
fresh_model.load_state_dict(best_state)
fresh_model.eval()

session_inputs = []
session_predictions = []

seg_statuses = {
    0:"Bronze",
    1:"Silver",
    2:"Gold",
    3:"Platnium"
}
while True:
    raw_name_input = input("-- Enter Customer Name ( or type 'exit' to stop ): ").strip()
    if raw_name_input.lower() == 'exit' or not raw_name_input:
        print("\nExiting Interactive Loop ...")
        break
        
    user_name = raw_name_input.title()
    years_val = float(input("-- Enter Years as a Customer : ").strip())
    spend_val = float(input("-- Enter Annual Spend ($) : ").strip())
    items_val = float(input("-- Enter Total Items Purchased : ").strip())
    return_val = float(input("-- Enter Return Rate (0.0 to 1.0) : ").strip())
    logins_val = float(input("-- Enter Monthly App Logins : ").strip())
    tickets_val = float(input("-- Enter Support Tickets Raised : ").strip())

    raw_array = np.array([[years_val, spend_val, items_val, return_val, logins_val, tickets_val]])
    scaled_tensor = torch.tensor(x_scaler.transform(raw_array), dtype=torch.float32).to(device)    
   
    with torch.inference_mode():
        logits = fresh_model(scaled_tensor)
        probability = torch.softmax(logits, dim=1)
        pred_class_idx = torch.argmax(probability, dim=1).item()

    string_prediction = encoder.inverse_transform([pred_class_idx])[0]
    print(f"Predicted Customer Segmentation Status = {seg_statuses.get(string_prediction)}")   

    session_inputs.append(f"{user_name} ( {spend_val} $ Spent )")
    session_predictions.append(probability * 100).detach().numpy()


onedrive = input("-- Do you use One Drive or Not ( Yes / No ) :").strip().title()
if onedrive == "Yes":
    desktop_path = f"C:/Users/{user_account}/OneDrive/Desktop/Customer Segmentation Status Multi Classifer Network Details.txt"
elif onedrive == "No":
    desktop_path = f"C:/Users/{user_account}/Desktop/Customer Segmentation Status Multi Classifer Network Details.txt"

try:
    with open(desktop_path, "w", encoding="utf-8") as file:
            file.write(f"== Customer Segmentation Status Multi Classifer Network Weights & Biases ==\n")   
            file.write(f"\n● Parameter Weights = {fresh_model.network[0].weight[0][0].item():.3f}\n")     
            file.write(f"● Parameter Bias = {fresh_model.network[0].bias[0].item():.3f}\n")     
            file.write(f"● Best Loss = {best_loss:.3f}\n")
            file.write(f"● Worst Loss = {max(losses):.3f}\n")
    print(f"\nSuccessfully Saved the Network Details")
except BaseException as e:
    print(f"Failed to Save Network Details = {e}")    



c:\Users\there\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Device = Cuda
X = (15000, 6)
Y = (15000,)
X Train = (12000, 6)
Y Train = (12000,)

== Search Leaderboard ==
Learning Rate   | Epochs   | Val Loss   | Val Acc % 


100%|██████████| 450/450 [00:06<00:00, 67.56it/s] 


0.01            | 450      | 0.0337     | 99.06632900238037 %


100%|██████████| 610/610 [00:07<00:00, 78.26it/s]


...

0.2             | 610      | 0.0715     | 98.1404721736908 %


100%|██████████| 790/790 [00:05<00:00, 156.86it/s]


0.2             | 790      | 0.0589     | 98.20830821990967 %


100%|██████████| 820/820 [00:07<00:00, 106.73it/s]


0.2             | 820      | 0.1268     | 97.64547348022461 %

● Best Loss Achieved = 0.0160    
● Worst Loss = 0.1268    
● Mean Accuracy = 98.75156953930855 %
● Precision Score = 0.9848
● Recall Score = 0.9765


Exiting Interactive Loop ...
Failed to Save Network Details = name 'desktop_path' is not defined
